In this notebook we are firstly checking, whether the responses are correct , and also loading the dataset from huggingface

In [10]:
import os, json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

sample_messages = [
    "HELP trapped on roof with family water rising fast near riverside apts zone A",
    "Just saw the news, hope everyone in the flood zone stays safe 🙏",
    "Main bridge on Highway 5 has completely collapsed, do not attempt to cross",
    "lol my basement is flooded again typical monday",
    "URGENT need insulin, diabetic elderly mother stuck no power zone C block 4",
]

In [11]:
def classify_raw(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": f"Is this an SOS/disaster-related message needing action? Explain briefly.\n\nMessage: \"{text}\""}
        ],
    )
    return response.choices[0].message.content

for msg in sample_messages:
    print(msg, "\n->", classify_raw(msg), "\n")

HELP trapped on roof with family water rising fast near riverside apts zone A 
-> Yes, this is an SOS/disaster-related message that clearly indicates an urgent situation requiring immediate action. The sender is trapped on a roof with their family, and there is a rising water threat near a riverside area, indicating a potential flooding emergency. This message signals a need for rescue and assistance to ensure the safety of those trapped. 

Just saw the news, hope everyone in the flood zone stays safe 🙏 
-> The message expresses concern for people's safety in a flood zone but does not request or suggest any specific action. It may indicate awareness of a disaster situation, but it appears more focused on offering support rather than needing immediate action. So, it does not qualify as an SOS or disaster-related message requiring action. 

Main bridge on Highway 5 has completely collapsed, do not attempt to cross 
-> Yes, this message is clearly an SOS/disaster-related communication tha

In [13]:
SOS_SCHEMA_PROMPT = """
Classify this message for disaster response. Return ONLY JSON matching this schema:

{{
  "is_actionable_sos": boolean,
  "disaster_type": "flood" | "fire" | "earthquake" | "storm" | "other" | "none",
  "severity": "low" | "medium" | "high" | "critical",
  "location_hint": string or null,
  "reason": "one short sentence"
}}

Message: "{text}"
"""

def classify_structured(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": SOS_SCHEMA_PROMPT.format(text=text)}],
    )
    return json.loads(response.choices[0].message.content)

results = [classify_structured(m) for m in sample_messages]
for msg, r in zip(sample_messages, results):
    print(msg, "\n->", r, "\n")


HELP trapped on roof with family water rising fast near riverside apts zone A 
-> {'is_actionable_sos': True, 'disaster_type': 'flood', 'severity': 'critical', 'location_hint': 'riverside apts zone A', 'reason': 'Family is trapped on a roof as water levels rise.'} 

Just saw the news, hope everyone in the flood zone stays safe 🙏 
-> {'is_actionable_sos': False, 'disaster_type': 'flood', 'severity': 'medium', 'location_hint': None, 'reason': 'The message expresses concern for those affected by a flood.'} 

Main bridge on Highway 5 has completely collapsed, do not attempt to cross 
-> {'is_actionable_sos': True, 'disaster_type': 'other', 'severity': 'high', 'location_hint': 'Highway 5', 'reason': 'The collapse of a main bridge poses a significant risk to safety.'} 

lol my basement is flooded again typical monday 
-> {'is_actionable_sos': True, 'disaster_type': 'flood', 'severity': 'medium', 'location_hint': 'basement', 'reason': 'The basement flooding indicates a potential water damage 

In [14]:
edge_cases = [
    "flood warning issued for tomorrow, stay tuned",
    "IM DROWNING in paperwork lol send help",
    "3 people trapped basement 45 elm street can't swim",
]
for msg in edge_cases:
    print(msg, "\n->", classify_structured(msg), "\n")

flood warning issued for tomorrow, stay tuned 
-> {'is_actionable_sos': False, 'disaster_type': 'flood', 'severity': 'medium', 'location_hint': None, 'reason': 'A warning has been issued but no immediate action is needed.'} 

IM DROWNING in paperwork lol send help 
-> {'is_actionable_sos': False, 'disaster_type': 'none', 'severity': 'low', 'location_hint': None, 'reason': 'The message is not a genuine distress signal.'} 

3 people trapped basement 45 elm street can't swim 
-> {'is_actionable_sos': True, 'disaster_type': 'flood', 'severity': 'high', 'location_hint': '45 elm street', 'reason': 'Three people are trapped and cannot swim.'} 



In [9]:
from datasets import load_dataset
ds = load_dataset("venetis/disaster_tweets")
sample = ds["train"][0]
sample

# Loading our dataset

README.md:   0%|          | 0.00/310 [00:00<?, ?B/s]

c:\Users\jenil\OneDrive\Desktop\Machine learning\llm_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jenil\.cache\huggingface\hub\datasets--venetis--disaster_tweets. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/7613 [00:00<?, ? examples/s]

{'id': 1,
 'keyword': None,
 'location': None,
 'text': 'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all',
 'target': 1}

In [18]:
import requests
url = "https://api.gdeltproject.org/api/v2/doc/doc" 
params = {
    "query": "flood disaster sourcelang:eng",
    "mode": "artlist",
    "maxrecords": 5,
    "format": "json",
}
resp = requests.get(url,params= params)
resp.json()

{'articles': [{'url': 'https://dailypioneer.com/news/joint-indian-army-and-sdrf-flood-relief-exercise-at-jorhat',
   'url_mobile': '',
   'title': 'Indian Army , SDRF Conduct  Exercise Jal Raksha III  Ahead of Monsoon Season in Assam',
   'seendate': '20260514T031500Z',
   'socialimage': 'https://dailypioneer.com/uploads/2026/story/images/big/joint-indian-army-and-sdrf-flood-relief-exercise-at-jorhat-2026-05-14.jpg',
   'domain': 'dailypioneer.com',
   'language': 'English',
   'sourcecountry': 'India'},
  {'url': 'https://www.newkerala.com/news/a/joint-indian-army-sdrf-flood-relief-exercise-jorhat-333.htm',
   'url_mobile': '',
   'title': 'Indian Army & SDRF Joint Flood Relief Exercise Jal Raksha III',
   'seendate': '20260514T071500Z',
   'socialimage': 'https://one.newkerala.com/images/t/newkerala-com-news250.webp',
   'domain': 'newkerala.com',
   'language': 'English',
   'sourcecountry': 'India'},
  {'url': 'https://timesofindia.indiatimes.com/city/hubballi/flood-mock-drill-held

In [19]:
def normalize_kaggle(record):
    return{
        "text":record["text"],
        "source":"kaggle_replay",
        "location_hint": record.get("location"),
        "timestamp": None,

    }

def normalize_gdelt(article):
    return{
        "text": article["title"],
        "source": "gdelt",
        "location_hint": None,
        "timestamp": None,
    }

# Basically in this cell we are normalizing the data i.e , alll data which we will receive should come in this format so that we can use it for our model training and testing.

In [20]:
kaggle_sample = normalize_kaggle(ds["train"][0])
gdelt_samples = [normalize_gdelt(a) for a in resp.json()["articles"][:3]]

for record in [kaggle_sample] + gdelt_samples:
    result = classify_structured(record["text"])
    print(record["source"], "|", record["text"][:80])
    print("->", result, "\n")

kaggle_replay | Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
-> {'is_actionable_sos': False, 'disaster_type': 'earthquake', 'severity': 'medium', 'location_hint': None, 'reason': 'This message references an earthquake but does not indicate a need for immediate action.'} 

gdelt | Indian Army , SDRF Conduct  Exercise Jal Raksha III  Ahead of Monsoon Season in 
-> {'is_actionable_sos': False, 'disaster_type': 'none', 'severity': 'low', 'location_hint': 'Assam', 'reason': 'The message refers to a preparedness exercise, not an actual disaster event.'} 

gdelt | Indian Army & SDRF Joint Flood Relief Exercise Jal Raksha III
-> {'is_actionable_sos': False, 'disaster_type': 'flood', 'severity': 'none', 'location_hint': None, 'reason': 'This message pertains to a relief exercise, not an active disaster situation.'} 

gdelt | Flood mock drill held at Ullal Lake to bolster disaster preparedness
-> {'is_actionable_sos': False, 'disaster_type': 'flood', 'severity': 'low', '

# Things I performed in this notebook :
Wrote classify_raw — a zero-shot baseline that judges whether a text message is an actionable SOS.

Wrote classify_structured — the real Alert Monitor output shape (is_actionable_sos, disaster_type, severity, location_hint, reason) as strict JSON.

Stress-tested edge cases (sarcasm, no-keyword real emergencies) to check the prompt holds up beyond obvious examples.

Loaded a Kaggle/HuggingFace disaster-tweets dataset as a repeatable ingestion source for development and future evals.

Queried GDELT for live disaster news, and fixed a language-filtering issue (sourcelang:eng) after noticing non-English results.

Wrote normalize() functions to convert both the dataset rows and GDELT articles into one common shape, proving the same classify_structured pipeline works regardless of source.